# fawp-index: Information-Control Exclusion Principle

**Ralph Clayton (2026)** · [doi:10.5281/zenodo.18673949](https://doi.org/10.5281/zenodo.18673949)

> *Your model still predicts. But you've already lost control.*

This notebook walks through the core concepts and API of `fawp-index`:

1. What is FAWP?
2. Reproduce the published E8 experimental figures
3. Run on your own data (DataFrame API)
4. Quant finance: crowded trade detection
5. Feature importance for ML engineers
6. Sklearn Pipeline integration

In [ ]:
# Install (uncomment if running in Colab)
# !pip install fawp-index[all] -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 120

import fawp_index
print(f'fawp-index version: {fawp_index.__version__}')
print(f'DOI: {fawp_index.__doi__}')

---
## 1. What is FAWP?

**Future Access Without Presence (FAWP)** is the condition where:
- Predictive coupling **persists** — the system still forecasts the future accurately
- Steering coupling has **collapsed** — the system can no longer influence the future

The **Information-Control Exclusion Principle** states: in unstable regimes, prediction and control are conjugate. When control fails, predictive information *surges* — just as it becomes uncapturable.

Let's generate a synthetic example to see it clearly.

In [ ]:
from fawp_index import FAWPAlphaIndex
from fawp_index.io.feeds import load_synthetic_demo

# Seismic example: zero control, maximum prediction
data = load_synthetic_demo('seismic', seed=42)
print(f'Loaded {len(data.pred_series):,} observations')
print(f'Source: {data.metadata["source"]}')

In [ ]:
result = FAWPAlphaIndex(
    eta=1e-4,       # minimum pred MI to gate FAWP
    epsilon=1e-4,   # maximum steer MI for FAWP
    n_null=200,
).compute(
    pred_series   = data.pred_series,
    future_series = data.future_series,
    action_series = data.action_series,
    obs_series    = data.obs_series,
    tau_grid      = list(range(1, 16)),
)

print(result.summary())

In [ ]:
# The leverage gap plot — prediction (orange) vs steering (blue)
fig, ax = result.plot(show=False)
plt.show()

---
## 2. Reproduce Published E8 Figures

The actual experimental data from Clayton (2026) Experiment E8 is **bundled with the package**.
This exactly reproduces the figures in the published paper.

**E8 parameters:** `a=1.02, K=0.8, Δ=20, 400 trials, x_fail=500`

In [ ]:
from fawp_index.data import E8_CONFIRM_FULL
import pandas as pd

df_e8 = pd.read_csv(E8_CONFIRM_FULL)
print(f'E8 data: {len(df_e8)} delay values, τ=0..{df_e8.delay.max()}')

# Key published result
peak = df_e8.loc[df_e8.mi_pred_strat.idxmax()]
print(f'Peak stratified pred MI: {peak.mi_pred_strat:.4f} bits at τ={int(peak.delay)}')
print(f'(Paper reports: 2.2337 bits at τ=9)')

In [ ]:
# Reproduce MICRO figure (τ=0..15) — the resonance ridge
from examples.reproduce_e8 import plot_e8_micro

df_micro = df_e8[df_e8.delay <= 15].copy()
fig, ax = plot_e8_micro(df_micro, save_path=None)
plt.show()

In [ ]:
# Full sweep (τ=0..80) — prediction persists long after control has failed
from examples.reproduce_e8 import plot_e8_leverage_gap

fig, ax = plot_e8_leverage_gap(df_e8, save_path=None)
plt.show()

---
## 3. DataFrame API — Your Own Data

Pass a pandas DataFrame directly. No numpy extraction needed.

In [ ]:
from fawp_index import fawp_from_dataframe, fawp_rolling

# Simulate financial data
np.random.seed(42)
n = 3000
signal = np.random.randn(n)         # factor signal
future = signal[21:] * 0.3 + np.random.randn(n-21) * 0.8   # forward return
volume = np.abs(np.random.randn(n)) + 1

df = pd.DataFrame({
    'factor_signal': signal[:n-21],
    'forward_return': future,
    'volume': volume[:n-21],
})
print(df.head())

In [ ]:
# Run FAWP directly on the DataFrame
result = fawp_from_dataframe(
    df,
    pred_col   = 'factor_signal',
    action_col = 'volume',
    future_col = 'forward_return',
    tau_grid   = list(range(1, 12)),
)
print(result.summary())

In [ ]:
# Rolling regime detection — adds fawp_* columns
df_regime = fawp_rolling(
    df,
    pred_col   = 'factor_signal',
    action_col = 'volume',
    future_col = 'forward_return',
    window = 500,
    step   = 50,
)

print(f'Rolling windows: {len(df_regime)}')
print(f'FAWP windows: {df_regime.fawp_in_regime.sum()}')

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(df_regime.index, df_regime.fawp_pred_mi, label='Pred MI', color='darkorange')
axes[0].plot(df_regime.index, df_regime.fawp_steer_mi, label='Steer MI', color='steelblue', linestyle='--')
axes[0].legend()
axes[0].set_title('Rolling FAWP — Predictive vs Steering MI')
axes[1].fill_between(df_regime.index, df_regime.fawp_in_regime.astype(int),
                     alpha=0.4, color='darkorange', label='FAWP regime')
axes[1].set_title('FAWP Regime Flag')
axes[1].legend()
plt.tight_layout()
plt.show()

---
## 4. Quant Finance: Crowded Trade Detection

In [ ]:
from fawp_index.quant import MomentumDecayDetector

np.random.seed(99)
n = 3000

# Scenario: momentum signal predicts returns well
# but trade size no longer moves the market (crowded)
momentum_signal = np.random.randn(n)
forward_return  = 0.4 * momentum_signal[:n-21] + np.random.randn(n-21) * 0.5
trade_size      = np.random.randn(n-21)           # we're trading...
price_impact    = np.random.randn(n-21) * 0.001   # ...but impact is near zero

detector = MomentumDecayDetector(tau_grid=list(range(1, 12)))
result = detector.detect(momentum_signal[:n-21], forward_return, trade_size, price_impact)
print(result.summary())

In [ ]:
from fawp_index.quant import EventStudyFAWP

# FAWP around earnings announcements (quarterly = every 63 bars)
prices = np.cumsum(np.random.randn(5000) * 0.01)
returns = np.diff(prices)
volumes = np.abs(np.random.randn(len(returns))) + 1
earnings_dates = list(range(252, len(returns)-50, 63))

study = EventStudyFAWP(pre_window=10, post_window=10)
ev_result = study.analyze(returns, volumes, earnings_dates)
print(ev_result.summary())

---
## 5. Feature Importance for ML Engineers

Which of your features are in the FAWP regime? i.e. which features predict well but cannot be acted upon?

In [ ]:
from fawp_index.features import FAWPFeatureImportance

np.random.seed(7)
n = 3000

# Build a DataFrame with multiple features
target = np.cumsum(np.random.randn(n) * 0.01)
df_ml = pd.DataFrame({
    'momentum':  np.roll(target, 21) + np.random.randn(n) * 0.01,  # lagged target — predictive
    'value':     np.random.randn(n),                                 # noise
    'quality':   np.random.randn(n) * 0.5,                          # noise
    'carry':     np.roll(target, 5) + np.random.randn(n) * 0.02,   # short lag — predictive
    'trade_vol': np.random.randn(n) * 0.001,                        # action proxy (tiny = FAWP)
})

fi = FAWPFeatureImportance(
    action_col = 'trade_vol',
    delta = 21,
    tau_grid = list(range(1, 8)),
    n_null = 100,
)

fi_result = fi.fit(df_ml, feature_cols=['momentum', 'value', 'quality', 'carry'])
print(fi_result.summary())

In [ ]:
fig, ax = fi_result.plot(show=False)
plt.show()

---
## 6. Sklearn Pipeline Integration

In [ ]:
from fawp_index.sklearn_api import FAWPTransformer

np.random.seed(42)
n = 3000

# X columns: [predictor, action]
X = np.column_stack([
    np.random.randn(n),        # factor signal
    np.random.randn(n) * 0.001 # action (tiny = FAWP)
])

fawp_t = FAWPTransformer(
    pred_col   = 0,
    action_col = 1,
    delta      = 20,
    tau_grid   = list(range(1, 10)),
    n_null     = 100,
)

fawp_t.fit(X)
print(f'FAWP score:   {fawp_t.score(X):.4f}')
print(f'In FAWP:      {fawp_t.in_fawp_}')
print(f'Agency τ_h:   {fawp_t.tau_h_}')

features = fawp_t.transform(X)
print(f'\nTransformed shape: {features.shape}')
print('Columns: [tau, pred_mi, steer_mi, alpha_index]')
print(features[:3])

In [ ]:
# Works in sklearn Pipeline
try:
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler

    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('fawp',   FAWPTransformer(pred_col=0, action_col=1, n_null=50)),
    ])

    score = pipe.fit(X).score(X)
    print(f'Pipeline FAWP score: {score:.4f}')
except ImportError:
    print('sklearn not installed — pip install scikit-learn')

---
## Summary

| API | Use case |
|-----|----------|
| `FAWPAlphaIndex` | Core detector, numpy arrays |
| `fawp_from_dataframe` | Pass a DataFrame directly |
| `fawp_rolling` | Annotate DataFrame with regime flags |
| `FAWPTransformer` | Sklearn Pipeline compatible |
| `FAWPFeatureImportance` | Which features are in FAWP regime? |
| `FAWPRegimeDetector` | Rolling market breakdown detection |
| `MomentumDecayDetector` | Crowded trade / execution edge collapse |
| `RiskParityWarning` | Vol-targeting failure early warning |
| `EventStudyFAWP` | Announcement study |
| `fawp-index` CLI | No Python required |

---

**Ralph Clayton (2026)** · [pypi.org/project/fawp-index](https://pypi.org/project/fawp-index/) · [doi:10.5281/zenodo.18673949](https://doi.org/10.5281/zenodo.18673949)